In [39]:
import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import shape
from pathlib import Path
import os
from bs4 import BeautifulSoup
import re

In [40]:
CURRENT_DIRECTORY =  os.getcwd()
SRC_DIRECTORY = Path(CURRENT_DIRECTORY).parents[1]
print(SRC_DIRECTORY)

BASE_DATASET_PATH = Path(SRC_DIRECTORY).parents[0]
BASE_DATASET_PATH = BASE_DATASET_PATH / "datasets"
print(BASE_DATASET_PATH)

/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/src
/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/datasets


In [41]:
education_2010_filepath = Path(BASE_DATASET_PATH / "singapore_data/singstat/2010_highest_education.xlsx")
education_2015_filepath = Path(BASE_DATASET_PATH / "singapore_data/data_gov/2015PlanningAreaHighestQualificationAttained.csv")
education_2020_filepath = Path(BASE_DATASET_PATH / "singapore_data/singstat/2020_highest_education.xlsx")

education_2010 = pd.read_excel(education_2010_filepath)
education_2015 = pd.read_csv(education_2015_filepath)
education_2020 = pd.read_excel(education_2020_filepath)


In [42]:
column_names = education_2020.columns.tolist()
new_column_names = [s.lower() for s in column_names]

print(column_names)

['planning area', 'Total', 'No Qualification', 'Primary', 'Lower Secondary', 'Secondary', 'Post-Secondary (Non-Tertiary)', 'Polytechnic Diploma', 'Professional Qualification and Other Diploma', 'University']


In [43]:
new_column_names = ['planning_area', 'total', 'no_qualification',
                    'pri', 'lower_sec', 'sec',
                    'post_sec_(non-tertiary)', 'poly_diploma',
                    'professional_qualification_&_other_diploma', 'uni']
education_2010.columns = new_column_names
# make the planning area names be lower case
education_2010['planning_area'] = education_2010['planning_area'].str.lower()
education_2010.head()

,planning_area,total,no_qualification,pri,lower_sec,sec,post_sec_(non-tertiary),poly_diploma,professional_qualification_&_other_diploma,uni
0,total,2779524,424443,193181,282523,526359,307562,250213,161144,634098
1,ang mo kio,140471,28602,11235,15798,24569,13592,11731,7162,27783
2,bedok,222290,33901,14542,22563,41999,24186,17438,12986,54675
3,bishan,68301,7486,3189,5256,12913,7417,5384,4621,22035
4,bukit batok,106726,14604,6677,9735,19178,11781,10103,6282,28365


In [44]:
education_2015.columns = new_column_names
# the numbers for 2015 are in thousands
# 1. Identify and select all columns with numeric data types (float or integer)
# We exclude the 'planning area' column which contains strings.
numeric_cols = education_2015.select_dtypes(include=np.number).columns

# 2. Multiply only the selected numeric columns by 1000
education_2015[numeric_cols] = education_2015[numeric_cols] * 1000

# Convert the results to integers 
education_2015[numeric_cols] = education_2015[numeric_cols].astype(int)

# make the planning area names be lower case
education_2015['planning_area'] = education_2015['planning_area'].str.lower()

education_2015.head()

,planning_area,total,no_qualification,pri,lower_sec,sec,post_sec_(non-tertiary),poly_diploma,professional_qualification_&_other_diploma,uni
0,total,2948500,368100,203400,236500,544800,304000,268100,215700,807900
1,ang mo kio,145200,23900,13400,13300,24600,13600,11100,10100,35400
2,bedok,212500,26700,15100,16300,40800,20700,16000,13900,62900
3,bishan,68100,6200,3100,4800,11800,6300,5500,5000,25400
4,bukit batok,105200,11800,7700,8500,19200,9900,11200,7200,29700


In [45]:
education_2020.columns = new_column_names
# make the planning area names be lower case
education_2020['planning_area'] = education_2020['planning_area'].str.lower()
education_2020.head()

,planning_area,total,no_qualification,pri,lower_sec,sec,post_sec_(non-tertiary),poly_diploma,professional_qualification_&_other_diploma,uni
0,total,3140436,333821,173912,258136,505564,341886,301963,217970,1007183
1,ang mo kio,132133,20625,9551,12060,21200,12397,10556,8508,37234
2,bedok,218749,23987,13335,15887,36058,22719,17306,14389,75068
3,bishan,69207,5379,2374,4739,9997,5993,5771,4458,30497
4,bukit batok,126128,12735,6954,10561,20076,13684,11727,8290,42101


In [27]:
def compare_rows(df_one, df_one_column: str, df_two, df_two_column: str):
    """
    To compare the rows of 2 dataframes to check for missing rows or mismatch row names

    Parameters
    ------
    - df_one: dataframe
        the first dataframe you want to compare

    - df_one_column: str
        name of the columns of the first dataframe you want to compare
    
    - df_two: dataframe
        the second dataframe you want to compare

    - df_two_column: str
        name of the columns of the second dataframe you want to compare

    Returns
    -------
    - missing areas of the first subzone
    
    - missing areas of the second subzone
    """

    first_subzones = df_one[df_one_column].tolist()
    second_subzones = df_two[df_two_column].tolist()

    # removing \n
    cleaned_first_subzones = [s.replace('\n', '') for s in first_subzones]
    cleaned_second_subzones = [s.replace('\n', '') for s in second_subzones]

    missing_areas_of_first_subzone = set(cleaned_second_subzones).difference(set(cleaned_first_subzones))
    print(len(cleaned_first_subzones))
    # print(missing_areas_of_first_subzone)

    missing_areas_of_second_subzone = set(cleaned_first_subzones).difference(set(cleaned_second_subzones))
    print(len(cleaned_second_subzones))
    # print(missing_areas_of_second_subzone)

    return missing_areas_of_first_subzone, missing_areas_of_second_subzone


In [46]:
missing_area_of_2010, missing_area_of_2015 = compare_rows(education_2010, "planning_area",
                                                          education_2015, "planning_area")
print(f"missing for 2010: {missing_area_of_2010}")
print(f"missing for 2015: {missing_area_of_2015}")

missing_area_of_2015, missing_area_of_2020 = compare_rows(education_2015, "planning_area",
                                                          education_2020, "planning_area")
print(f"missing for 2015: {missing_area_of_2015}")
print(f"missing for 2020: {missing_area_of_2020}")

37
30
missing for 2010: set()
missing for 2015: {'singapore river', 'river valley', 'newton', 'changi', 'mandai', 'rochor', 'downtown core'}
30
32
missing for 2015: {'downtown core', 'river valley'}
missing for 2020: set()
